# Marketing Agent — SFT → DPO → Eval (Colab)

End-to-end training and evaluation for **Preference Optimization for Marketing Agents**.

**Before you start:**
1. `Runtime → Change runtime type → GPU` (T4 is enough for a 3B model in 4-bit).
2. Run Phases 1-3 locally first and **push** `data/preferences/preferences.jsonl` — cloning the repo brings it in.
3. (Optional) Have your `GEMINI_API_KEY` ready for the judging step at the end.


## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Clone the repo + install training deps

Replace the URL if you forked it elsewhere.

In [ ]:
REPO_URL = 'https://github.com/SalahnAI/Preference-Optimization-for-Marketing-Agents.git'

import os
if not os.path.isdir('Preference-Optimization-for-Marketing-Agents'):
    !git clone $REPO_URL
%cd Preference-Optimization-for-Marketing-Agents

In [ ]:
!pip install -q -r training/requirements-train.txt

## Hugging Face auth (optional, recommended)

Qwen2.5 is **ungated** so anonymous downloads work, but a token gives faster,
rate-limit-free downloads and unlocks gated models (e.g. Llama). Get one at
https://huggingface.co/settings/tokens (read scope). Press Enter to skip.

In [ ]:
import os, getpass
tok = getpass.getpass('HF token (Enter to skip): ').strip()
if tok:
    os.environ['HF_TOKEN'] = tok
    from huggingface_hub import login
    login(tok)
    print('HF auth set')
else:
    print('skipped — anonymous download (fine for Qwen2.5)')

## 2. Sanity-check the preference dataset

If this errors, run the local Phases 1-3 and push `preferences.jsonl` first.

In [ ]:
import json, pathlib
p = pathlib.Path('data/preferences/preferences.jsonl')
assert p.exists(), 'preferences.jsonl missing — run Phases 1-3 locally and push it.'
rows = [json.loads(l) for l in p.open()]
print(f'{len(rows)} preference pairs')
print('keys:', list(rows[0].keys()))
print('\n--- example chosen ---\n', rows[0]['chosen'][:400])

## 3. Pick the base model

Qwen 2.5 3B by default; swap for `meta-llama/Llama-3.2-3B-Instruct` (gated — needs HF login).

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

## 4. Phase 5 — SFT (LoRA on the *chosen* responses)

~10-20 min on a T4 for a few hundred pairs.

In [ ]:
!python training/sft_train.py \
    --model $BASE_MODEL \
    --data data/preferences/preferences.jsonl \
    --out training/outputs/sft \
    --epochs 2

## 5. Phase 6 — DPO (starts from the SFT adapter)

In [ ]:
!python training/dpo_train.py \
    --base $BASE_MODEL \
    --sft_adapter training/outputs/sft \
    --data data/preferences/preferences.jsonl \
    --out training/outputs/dpo \
    --epochs 1 --beta 0.1

## 6. Phase 7a — Generate predictions on the held-out set

Three variants on the same disjoint campaigns (different seed → never trained on).

In [ ]:
!python evaluation/infer.py --base $BASE_MODEL --tag base
!python evaluation/infer.py --base $BASE_MODEL --adapter training/outputs/sft --tag sft
!python evaluation/infer.py --base $BASE_MODEL --adapter training/outputs/dpo --tag dpo

In [ ]:
# Peek at one base-vs-dpo pair
import json
b = [json.loads(l) for l in open('results/preds_base.jsonl')]
d = [json.loads(l) for l in open('results/preds_dpo.jsonl')]
print('PROMPT:\n', b[0]['prompt'][:300])
print('\n=== BASE ===\n', b[0]['output'][:500])
print('\n=== DPO ===\n', d[0]['output'][:500])

## 7. Phase 7b — Judge the outputs (needs GEMINI_API_KEY)

Win rate + per-axis quality scores. Writes `results/eval_report.md`.
You can also skip this and run `run_eval.py` locally instead.

In [ ]:
import getpass, os
os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')
!pip install -q google-genai python-dotenv

In [ ]:
# DPO vs base (the headline number)
!python evaluation/run_eval.py --a results/preds_base.jsonl --b results/preds_dpo.jsonl

In [ ]:
# Optional: isolate the DPO lift over SFT alone
!python evaluation/run_eval.py --a results/preds_sft.jsonl --b results/preds_dpo.jsonl

In [ ]:
from IPython.display import Markdown
Markdown(open('results/eval_report.md').read())

## 8. Save your work

Adapters and results are small. Either download them or commit/push back to the repo.

In [ ]:
# Download the trained adapters + report
!cd training/outputs && zip -qr /content/adapters.zip sft dpo
from google.colab import files
files.download('/content/adapters.zip')
files.download('results/eval_report.md')
files.download('results/eval_report.json')